# 01 · Del dato recibido al dato fiable (18–35 min)

Objetivo: reconciliar todas las recepciones y explicar por qué se acepta o rechaza cada registro. No deduplicar por máquina. Esta exploración trabaja en memoria y no publica aún resultados.

In [ ]:
import json
import pandas as pd
from industrialdata.common import root
from industrialdata.catalog import load_catalog
from industrialdata.ingest import ingest, business_tables
folder = root()
catalog = load_catalog(folder)  # Consulta PostgreSQL: NO el catálogo JSON de pruebas.
telemetry, report, rejects, files = ingest(folder / "raw", catalog, "exploracion-mini")
display(pd.DataFrame([report]))
assert report == {"input": 45, "accepted_new": 40, "duplicates": 1, "rejected": 4}
print("45 = 40 + 1 + 4")

In [ ]:
display(pd.DataFrame(rejects)[["source_file", "source_line", "reason", "raw"]])
display(telemetry[["sensor_id", "timestamp_utc", "value", "unit", "source_file", "source_line"]].head(10))

## Inspecciona la solución A1–A2
Un valor fuera del rango operativo puede aceptarse; uno fuera del rango físico se rechaza. Ante un conflicto se conserva la primera observación aceptada y se registra la posterior. Un dato recibido tarde no es necesariamente inválido.

In [ ]:
import inspect
from industrialdata import reglas
print(inspect.getsource(reglas.normalizar))
print(inspect.getsource(reglas.clasificar))

## Sigue una observación hasta el original
La procedencia contiene archivo, línea, hash y primera ejecución. Un resultado sin trazabilidad es difícil de defender.

In [ ]:
sample = telemetry[(telemetry.sensor_id == "M01_temperature") & (telemetry.value == 85)].iloc[0]
original = (folder / "raw" / sample.source_file).read_text().splitlines()[int(sample.source_line) - 1]
print(original)
assert json.loads(original)["value"] == 85

**Microejercicio en directo:** muestra el duplicado y el conflicto (líneas 41 y 42 del JSONL). ¿Por qué las dos recepciones no tienen el mismo tratamiento? No edites el original: explica la decisión usando las funciones.